<a href="https://colab.research.google.com/github/komalsathvik/DeepLearning/blob/main/CNN/CNN_imgclassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CNN FOR CIFAR10


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10


In [ ]:
#Datasets and DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
#transforms is helped for performed transformations on images
#image => scale(0,1) => normalize(-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train=CIFAR10(root='./data',train=True,download=True,transform=transform)
test=CIFAR10(root='./data',train=False,download=True,transform=transform)

In [ ]:
trainloader=DataLoader(train,batch_size=64,shuffle=True)
testloader=DataLoader(test,batch_size=64,shuffle=False)




> **BUILD THE CNN**



In [ ]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.conv_layers=nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2), #kernel size=2 and stride=2

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),
    )

    self.fc_layers=nn.Sequential(
        nn.Linear(128*4*4,256),
        nn.ReLU(),
        nn.Linear(256,10),
    )
  def forward(self,x):
    x=self.conv_layers(x)
    x=x.view(x.size(0),-1) #flattening step
    x=self.fc_layers(x)
    return x

In [ ]:
model=CNN()

In [ ]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)


**Training the CNN**

In [ ]:

epochs=10
for epoch in range(epochs):
  epoch_training_loss=0.0
  for images,labels in trainloader:
    optimizer.zero_grad()
    outputs=model(images)
    loss=criterion(outputs,labels)
    loss.backward()
    optimizer.step()
    epoch_training_loss+=loss.item()
  print(f"Epoch {epoch+1}/{epochs}, Training Loss: {epoch_training_loss/len(trainloader)}")

Epoch 1/10, Training Loss: 1.3855952722642122
Epoch 2/10, Training Loss: 0.9422074422964355
Epoch 3/10, Training Loss: 0.7496535248692383
Epoch 4/10, Training Loss: 0.6181239563485851
Epoch 5/10, Training Loss: 0.5161218562013353
Epoch 6/10, Training Loss: 0.42417983153401434
Epoch 7/10, Training Loss: 0.3332039007292989
Epoch 8/10, Training Loss: 0.2530528455258102
Epoch 9/10, Training Loss: 0.1887607987770034
Epoch 10/10, Training Loss: 0.15180678625145685


In [ ]:
#Evaluating our CNN

correct_labels=0
total_labels=0
model.eval()
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total_labels += labels.size(0)
        correct_labels += (predicted == labels).sum().item()
print(f"accuracy = {(correct_labels)/(total_labels)*100}")

accuracy = 74.86
